# Homework 4

**Name:** -- Federico Garcia Rodriguez --

**e-mail:** -- federico.garcia0747@alumnos.udg.mx --

**code:** -- 224807479 --

# Modulo

In [385]:
import numpy as np
import pandas as pd
import math
from scipy.stats import levy_stable

import panel as pn
pn.extension()
import panel.widgets as pnw

import plotly.graph_objects as go
pn.extension('plotly')

# Classes

In [496]:
class Vec2d(object):
    """2d vector class, supports vector and scalar operators,
       and also provides a bunch of high level functions
       """
    __slots__ = ['x', 'y']

    def __init__(self, x_or_pair, y = None):
        if y == None:            
            self.x = x_or_pair[0]
            self.y = x_or_pair[1]
        else:
            self.x = x_or_pair
            self.y = y
            
    # Addition
    def __add__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x + other.x, self.y + other.y)
        elif hasattr(other, "__getitem__"):
            return Vec2d(self.x + other[0], self.y + other[1])
        else:
            return Vec2d(self.x + other, self.y + other)

    # Subtraction
    def __sub__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x - other.x, self.y - other.y)
        elif (hasattr(other, "__getitem__")):
            return Vec2d(self.x - other[0], self.y - other[1])
        else:
            return Vec2d(self.x - other, self.y - other)
    
    # Vector length
    def get_length(self):
        return math.sqrt(self.x**2 + self.y**2)
    
    # rotate vector
    def rotated(self, angle):        
        cos = math.cos(angle)
        sin = math.sin(angle)
        x = self.x*cos - self.y*sin
        y = self.x*sin + self.y*cos
        return Vec2d(x, y)

# Functions

In [497]:
#####################################################################################
# levy Flight trajectoy
#####################################################################################
def levy_2d(n_steps=1000, speed=5, s_pos=[0,0], loc=0, alpha=1, beta=1, step_scale=1):
    """
    Arguments:
        n_steps:
        speed:
        s_pos:
        alpha:
        beta:
    Returns:
        levy_2d
    """
    positions = [s_pos]
    for _ in range(n_steps - 1):
        step_length = step_scale * (1 + np.random.pareto(alpha))**beta
        turning_angle = np.random.uniform(-np.pi, np.pi)
        new_x = positions[-1][0] + step_length * math.cos(turning_angle)+1
        new_y = positions[-1][1] + step_length * math.sin(turning_angle)+1
        positions.append([new_x, new_y])
    
    levy_2d = pd.DataFrame(positions, columns=['x_pos', 'y_pos'])
    return levy_2d

def step_lengths(trajectory):
    """
    Parameters:
        trajectory:

    Returns:
        step_lengths
    """
    dx = trajectory['x_pos'].diff()
    dy = trajectory['y_pos'].diff()

    step_lengths = np.sqrt(dx**2 + dy**2)
    return step_lengths 

#####################################################################################
# Correlated Random Walk motion trajectoy
#####################################################################################
def crw_2d(n_steps=1000, speed=5, s_pos=[0,0], cauchy = 0.9):
    """
    Arguments:
        n_steps:
        speed:
        s_pos:
        cauchy:
    Returns:
        CRW_2d_df
    """
    CRW_exponent = cauchy

    partpos = [s_pos]
    current_angle = 0  # initial direction
    for _ in range(n_steps - 1):
        # Draw turning angle from a Cauchy distribution
        turning_angle = CRW_exponent * np.random.standard_cauchy()
        current_angle += turning_angle
        add_pos = [partpos[-1][0] + speed * math.cos(current_angle),
                   partpos[-1][1] + speed * math.sin(current_angle)]
        partpos.append(add_pos)
    CRW_2d_df = pd.DataFrame(partpos, columns=['x_pos', 'y_pos'])
    return CRW_2d_df

#####################################################################################
# Brownian motion trajectoy
#####################################################################################
def bm_2d(n_steps=1000, speed=6, s_x_pos=0, s_y_pos=0):
    """
    Arguments:
        n_steps: 
        speed: 
        s_pos: 
    Returns:
        BM_2d_df: 
    """
    # Init velocity vector
    velocity = Vec2d(speed,0)

    BM_2d_df = pd.DataFrame(columns = ['x_pos','y_pos'])
    temp_df = pd.DataFrame([{'x_pos': s_x_pos, 'y_pos': s_y_pos}])
    BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)


    for i in range(n_steps-1):        
        #turn_angle = np.random.choice([0, np.pi/2, np.pi, 3*np.pi/2])
        turn_angle = np.random.uniform(low=-np.pi, high=np.pi)
        velocity = velocity.rotated(turn_angle)

        temp_df = pd.DataFrame([{'x_pos': BM_2d_df.x_pos[i]+velocity.x, 'y_pos': BM_2d_df.y_pos[i]+velocity.y}])
        BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)

    return BM_2d_df

# DASHBOARD

## Widgets

In [529]:
#labels
str_pane1 = pn.pane.Str(
    'RW Type',
    styles={'font-size': '12pt'}
)
str_pane2 = pn.pane.Str(
    'Parameters',
    styles={'font-size': '12pt'}
)
str_pane3 = pn.pane.Str(
    'Metrics',
    styles={'font-size': '12pt'}
)
str_pane4 = pn.pane.Str(
    '3D trajectory',
    styles={'font-size': '12pt'}
)
str_pane5 = pn.pane.Str(
    'Metrics',
    styles={'font-size': '12pt'}
)

#RW options
radio_group = pnw.RadioButtonGroup(
    name='RB_RW_options', options=['BM', 'CRW', 'LF'], button_type='light')

#radio_group_m = pnw.RadioButtonGroup(
    #name='RB_M_options', options=['PL', 'MSD'], button_type='light')

#Parameters
#BM
bm_input_steps = pnw.IntInput(name='Number of steps', value=500, step=100, start=0, end=1000)
bm_input_speed = pnw.IntInput(name='Speed', value=1, step=1, start=1, end=10)
bm_input_start_x = pnw.FloatInput(name='Starting pos x', value=0, step=1, start=0, end=100)
bm_input_start_y = pnw.FloatInput(name='Starting pos y', value=0, step=1, start=0, end=100)

#CRW 
crw_input_steps = pnw.IntInput(name='Number of steps', value=500, step=100, start=0, end=1000)
crw_input_speed = pnw.IntInput(name='Speed', value=1, step=1, start=1, end=10)
crw_input_start_x = pnw.FloatInput(name='Starting pos x', value=0, step=1, start=0, end=100)
crw_input_start_y = pnw.FloatInput(name='Starting pos y', value=0, step=1, start=0, end=100)
crw_input_cauchy = pnw.FloatInput(name='Cauchy coefficient', value=0.1, step=0.1, start=0.1, end=1)

#LF
lf_input_steps = pnw.IntInput(name='Number of steps', value=500, step=100, start=0, end=1000)
lf_input_speed = pnw.IntInput(name='Speed', value=1, step=1, start=1, end=10)
lf_input_start_x = pnw.FloatInput(name='Starting pos x', value=0, step=1, start=0, end=100)
lf_input_start_y = pnw.FloatInput(name='Starting pos y', value=0, step=1, start=0, end=100)
lf_input_alpha = pnw.FloatInput(name='Alpha coefficient', value=0.1, step=0.1, start=0.1, end=1)
lf_input_beta = pnw.FloatInput(name='Beta coefficient', value=0.1, step=0.1, start=0.1, end=1)

#Metrics selector
Metrics_selector = pnw.Select(name='Metrics type', options=['MSD', 'PL'], value='PL')

In [530]:
#column paremeters
column_bm_par = pn.Column(bm_input_steps, bm_input_speed, bm_input_start_x, bm_input_start_y)
column_crw_par = pn.Column(crw_input_steps, crw_input_speed, crw_input_start_x, crw_input_start_y, crw_input_cauchy)
column_lf_par = pn.Column(lf_input_steps, lf_input_speed, lf_input_start_x, lf_input_start_y, lf_input_alpha, lf_input_beta)

#column layout
@pn.depends(radio_group)
def rw_select(radio_group):
    match radio_group:
        case 'BM': column = column_bm_par
        case 'CRW': column = column_crw_par
        case 'LF': column = column_lf_par
    return column

@pn.depends(radio_group)
def plot_select(radio_group):
    match radio_group:
        case 'BM': traj = plot_bmtraj
        case 'CRW': traj = plot_crwtraj
        case 'LF': traj = plot_levytraj
            
    return traj

@pn.depends(radio_group_m)
def plot_metric(radio_group_m):
    match radio_group_m:
        case 'PL': traj = plot_bmpath
        case 'MSD': traj = plot_bmmsd
            
    return traj

@pn.depends(radio_group, Metrics_selector)
def metric_select(radio_group, Metrics_selector):
    match radio_group:
        case 'BM': 
                if Metrics_selector=='PL':
                    metric = plot_bmpath
                else:
                    metric = plot_bmmsd
        case 'CRW':
                if Metrics_selector=='PL':
                    metric = plot_crwpath
                else:
                    metric = plot_crwmsd
        case 'LF':
                if Metrics_selector=='PL':
                    metric = plot_levypath
                else:
                    metric = plot_levymsd      
    return metric

# Decorators

# Brownian Motion

In [531]:
@pn.depends(bm_input_steps, bm_input_speed, bm_input_start_x, bm_input_start_y)
def plot_bmtraj(bm_input_steps, bm_input_speed, bm_input_start_x, bm_input_start_y):
    bm_df = bm_2d(n_steps=bm_input_steps, speed=bm_input_speed, s_x_pos=bm_input_start_x, s_y_pos=bm_input_start_y)

    # Init figure
    fig_BM_3d = go.Figure()
    
    # Plot trajectory
    fig_BM_3d.add_trace(go.Scatter3d(x=bm_df.x_pos,
                                     y=bm_df.y_pos,
                                     z=bm_df.index,
                                     marker = dict(size=2),
                                     line = dict(width=2),
                                     mode = 'lines',
                                     name = 'BM 3d',
                                     showlegend = True))
    
    # Update ploy layout
    fig_BM_3d.update_layout(title_text = 'BM 3d',
                            autosize = True,
                            width=500,
                            height=500,
                            scene_camera=dict(
                                up = dict(x=1,y=1,z=1),
                                center = dict(x=0,y=0,z=0),
                                eye=dict(x=1,y=1,z=0.5),
                                projection=dict(type="orthographic")
                            )
    )
    return fig_BM_3d

#####################################################################################
# trajectory lenght path
#####################################################################################
def path_lenght(trajectory):
    from scipy.spatial import distance
    """
    Arguments:
        trajectory: input trajectory to get the path lenght
    Returns:
        path_lenght: cumulative path lenght
    """
    distance = np.array([distance.euclidean(trajectory.iloc[i-1], trajectory.iloc[i]) for i in range(1, trajectory.shape[0])])
    path_lenght = np.cumsum(distance)
    
    return path_lenght

@pn.depends(bm_input_steps, bm_input_speed, bm_input_start_x, bm_input_start_y)
def plot_bmpath(bm_input_steps, bm_input_speed, bm_input_start_x, bm_input_start_y):
    bm_df = bm_2d(n_steps=bm_input_steps, speed=bm_input_speed, s_x_pos=bm_input_start_x, s_y_pos=bm_input_start_y)
    pl_BM = path_lenght(bm_df)

    # Init figure
    fig_pathLenght_2d = go.Figure()
    
    # Plot trajectory
    fig_pathLenght_2d.add_trace(go.Scatter(x = np.arange(bm_input_steps),
                               y = pl_BM,
                               marker = dict(size=2),
                               line = dict(width = 2),
                               mode = 'lines',
                               name = 'path length BM',
                               showlegend = True))
    return fig_pathLenght_2d

#####################################################################################
# Mean Squared Displacement
#####################################################################################
def mean_squared_displacement(trajectory):
    """
    Arguments:
        trajectory
    Returns:
        msd
    """
    lenght = len(trajectory)
    msd = np.zeros_like(trajectory.x_pos)
    for delta in range(1, lenght):
        for i in range(delta,lenght):
            msd[delta] += ((trajectory.x_pos[i] - trajectory.x_pos[i-delta])**2 + (trajectory.y_pos[i] - trajectory.y_pos[i-delta])**2)
        msd[delta] = msd[delta] / (lenght - delta)
    return msd

@pn.depends(bm_input_steps, bm_input_speed, bm_input_start_x, bm_input_start_y)
def plot_bmmsd(bm_input_steps, bm_input_speed, bm_input_start_x, bm_input_start_y):
    bm_df = bm_2d(n_steps=bm_input_steps, speed=bm_input_speed, s_x_pos=bm_input_start_x, s_y_pos=bm_input_start_y)
    msd_BM = mean_squared_displacement(bm_df)

    # Init figure
    fig_msd_2d = go.Figure()
    
    # Plot trajectory
    fig_msd_2d.add_trace(go.Scatter(x = np.arange(bm_input_steps),
                               y = msd_BM,
                               marker = dict(size=2),
                               line = dict(width = 2),
                               mode = 'lines',
                               name = 'MSD BM',
                               showlegend = True))
    return fig_msd_2d

## Correlated Random Walk

In [532]:
@pn.depends(crw_input_steps, crw_input_start_x, crw_input_start_y, crw_input_speed, crw_input_cauchy)
def plot_crwtraj(crw_input_steps, crw_input_start_x, crw_input_start_y, crw_input_speed,crw_input_cauchy):
    crw_df = crw_2d(n_steps=crw_input_steps, speed=crw_input_speed, s_pos=[crw_input_start_x, crw_input_start_y],cauchy=crw_input_cauchy)

    # Init figure
    fig_CRW_3d = go.Figure()
    
    # Plot trajectory
    fig_CRW_3d.add_trace(go.Scatter3d(x=crw_df.x_pos,
                                     y=crw_df.y_pos,
                                     z=crw_df.index,
                                     marker = dict(size=2),
                                     line = dict(width=2),
                                     mode = 'lines',
                                     name = 'CRW 3d',
                                     showlegend = True))
    
    # Update ploy layout
    fig_CRW_3d.update_layout(title_text = 'CRW 3d',
                            autosize = True,
                            width=500,
                            height=500,
                            scene_camera=dict(
                                up = dict(x=1,y=1,z=1),
                                center = dict(x=0,y=0,z=0),
                                eye=dict(x=1,y=1,z=0.5),
                                projection=dict(type="orthographic")
                            )
    )
    return fig_CRW_3d

@pn.depends(crw_input_steps, crw_input_speed, crw_input_start_x, crw_input_start_y, crw_input_cauchy)
def plot_crwpath(crw_input_steps, crw_input_speed, crw_input_start_x, crw_input_start_y, crw_input_cauchy):
    crw_df = crw_2d(n_steps=crw_input_steps, speed=crw_input_speed, s_pos=[crw_input_start_x, crw_input_start_y],cauchy=crw_input_cauchy)
    pl_CRW = path_lenght(crw_df)

    # Init figure
    fig_pathLenght_2d = go.Figure()
    
    # Plot trajectory
    fig_pathLenght_2d.add_trace(go.Scatter(x = np.arange(crw_input_steps),
                               y = pl_CRW,
                               marker = dict(size=2),
                               line = dict(width = 2),
                               mode = 'lines',
                               name = 'path length CRW',
                               showlegend = True))
    return fig_pathLenght_2d

@pn.depends(crw_input_steps, crw_input_speed, crw_input_start_x, crw_input_start_y, crw_input_cauchy)
def plot_crwmsd(crw_input_steps, crw_input_speed, crw_input_start_x, crw_input_start_y,crw_input_cauchy):
    crw_df = crw_2d(n_steps=crw_input_steps, speed=crw_input_speed, s_pos=[crw_input_start_x, crw_input_start_y],cauchy=crw_input_cauchy)
    msd_CRW = mean_squared_displacement(crw_df)

    # Init figure
    fig_msd_2d = go.Figure()
    
    # Plot trajectory
    fig_msd_2d.add_trace(go.Scatter(x = np.arange(crw_input_steps),
                               y = msd_CRW,
                               marker = dict(size=2),
                               line = dict(width = 2),
                               mode = 'lines',
                               name = 'MSD CRW',
                               showlegend = True))
    return fig_msd_2d

## Levy Flight

In [533]:
@pn.depends(lf_input_steps, lf_input_speed,lf_input_start_x, lf_input_start_y,lf_input_alpha,lf_input_beta)
def plot_levytraj(lf_input_steps, lf_input_speed, lf_input_start_x, lf_input_start_y,lf_input_alpha, lf_input_beta):
    levy_df = levy_2d(n_steps=lf_input_steps, speed=lf_input_speed, s_pos=[lf_input_start_x, lf_input_start_y], alpha=lf_input_alpha, beta=lf_input_beta)

    # Init figure
    fig_LF_3d = go.Figure()
    
    # Plot trajectory
    fig_LF_3d.add_trace(go.Scatter3d(x=levy_df.x_pos,
                                     y=levy_df.y_pos,
                                     z=levy_df.index,
                                     marker = dict(size=2),
                                     line = dict(width=2),
                                     mode = 'lines',
                                     name = 'Levy Flight 3d',
                                     showlegend = True))
    
    # Update ploy layout
    fig_LF_3d.update_layout(title_text = 'Levy Flight in 3d',
                            autosize = True,
                            width=500,
                            height=500,
                            scene_camera=dict(
                                up = dict(x=1,y=1,z=1),
                                center = dict(x=0,y=0,z=0),
                                eye=dict(x=1,y=1,z=0.5),
                                projection=dict(type="orthographic")
                            )
    )
    return fig_LF_3d

@pn.depends(lf_input_steps, lf_input_speed,lf_input_start_x, lf_input_start_y,lf_input_alpha,lf_input_beta)
def plot_levypath(lf_input_steps, lf_input_speed, lf_input_start_x, lf_input_start_y,lf_input_alpha, lf_input_beta):
    levy_df = levy_2d(n_steps=lf_input_steps, speed=lf_input_speed, s_pos=[lf_input_start_x, lf_input_start_y], alpha=lf_input_alpha, beta=lf_input_beta)
    pl_levy = path_lenght(levy_df)

    # Init figure
    fig_pathLenght_2d = go.Figure()
    
    # Plot trajectory
    fig_pathLenght_2d.add_trace(go.Scatter(x = np.arange(lf_input_steps),
                               y = pl_levy,
                               marker = dict(size=2),
                               line = dict(width = 2),
                               mode = 'lines',
                               name = 'path length Levy Flight',
                               showlegend = True))
    return fig_pathLenght_2d

@pn.depends(lf_input_steps, lf_input_speed,lf_input_start_x, lf_input_start_y,lf_input_alpha,lf_input_beta)
def plot_levymsd(lf_input_steps, lf_input_speed, lf_input_start_x, lf_input_start_y,lf_input_alpha, lf_input_beta):
    levy_df = levy_2d(n_steps=lf_input_steps, speed=lf_input_speed, s_pos=[lf_input_start_x, lf_input_start_y], alpha=lf_input_alpha, beta=lf_input_beta)
    msd_Levy = mean_squared_displacement(levy_df)

    # Init figure
    fig_msd_2d = go.Figure()
    
    # Plot trajectory
    fig_msd_2d.add_trace(go.Scatter(x = np.arange(lf_input_steps),
                               y = msd_Levy,
                               marker = dict(size=2),
                               line = dict(width = 2),
                               mode = 'lines',
                               name = 'MSD Levy Flight',
                               showlegend = True))
    return fig_msd_2d

## Deploy Dashboard

In [535]:
column_params = pn.Column(str_pane1, radio_group, str_pane2, rw_select, str_pane3, Metrics_selector, styles=dict(background='WhiteSmoke'))
column_traj = pn.Column(str_pane4, plot_select)
column_metr = pn.Column(str_pane5, metric_select)

row = pn.Row(column_params, column_traj, column_metr, styles=dict(background='WhiteSmoke'))

server=row.show()

Launching server at http://localhost:63294
